# 01: Translation run

The **only** notebook that calls the LLM. For each cell in `RUN_MATRIX` it drives `SQLTranslator` over every gold query and writes one record per query to `records_<dataset>_<target>_<model>.json` in `evaluation/outputs/`. Downstream notebooks glob those files.

Configuration lives in `eval_harness.config` (temperature 0.0, max 3 iterations, syntax validation for Cypher). Token counts come straight from `result.token_usage` -- no log scraping. Resume is automatic: query ids already on disk for a cell are skipped.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "evaluation"))

from dataclasses import replace
from eval_harness import RUN_MATRIX, run_translation, load_records, OUTPUTS_DIR

for rc in RUN_MATRIX:
    print(rc)

RunConfig(dataset='ldbc', target='cypher', model='llama3.2:latest', provider='ollama', validation_mode=None, max_iterations=3, temperature=0.0, num_ctx=16384, host='http://localhost:11434', outputs_dir=PosixPath('/Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs'), subset=None, resume=True, server_config=None)
RunConfig(dataset='ldbc', target='cypher', model='qwen3-coder:30b', provider='ollama', validation_mode=None, max_iterations=3, temperature=0.0, num_ctx=16384, host='http://localhost:11434', outputs_dir=PosixPath('/Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs'), subset=None, resume=True, server_config=None)
RunConfig(dataset='ldbc', target='cypher', model='gemma4:26b', provider='ollama', validation_mode=None, max_iterations=3, temperature=0.0, num_ctx=16384, host='http://localhost:11434', outputs_dir=PosixPath('/Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs'), subset=None, resume=True, server_config=None)
RunConfig(da

## Optional smoke subset

Uncomment to restrict every matrix cell to one query before paying for a full run.

In [2]:
# RUN_MATRIX = [replace(rc, subset=('ldbc_q01',)) for rc in RUN_MATRIX]
print('Will run', len(RUN_MATRIX), 'matrix cell(s).')

Will run 12 matrix cell(s).


## Run

Writes incrementally; safe to interrupt and resume.

In [3]:
for rc in RUN_MATRIX:
    run_translation(rc)

Resume: 14 record(s) on disk for records_ldbc_cypher_llama3.2_latest.json; 14 query id(s) done.
ldbc/cypher/llama3.2:latest: 0 item(s) to translate.
Done: 14 record(s) in /Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs/records_ldbc_cypher_llama3.2_latest.json
Resume: 14 record(s) on disk for records_ldbc_cypher_qwen3-coder_30b.json; 14 query id(s) done.
ldbc/cypher/qwen3-coder:30b: 0 item(s) to translate.
Done: 14 record(s) in /Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs/records_ldbc_cypher_qwen3-coder_30b.json
Resume: 14 record(s) on disk for records_ldbc_cypher_gemma4_26b.json; 14 query id(s) done.
ldbc/cypher/gemma4:26b: 0 item(s) to translate.
Done: 14 record(s) in /Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs/records_ldbc_cypher_gemma4_26b.json
Resume: 14 record(s) on disk for records_ldbc_cypher_claude-opus-4-8.json; 14 query id(s) done.
ldbc/cypher/claude-opus-4-8: 0 item(s) to translate.
Done: 14 record(s) in /

ldbc/gremlin/claude-opus-4-8: 0 item(s) to translate.
Done: 14 record(s) in /Users/ivona.obonova/school/rows2graph/rows2graph/evaluation/outputs/records_ldbc_gremlin_claude-opus-4-8.json


## Run-level summary

In [4]:
import pandas as pd

df = pd.DataFrame(load_records(OUTPUTS_DIR))
if len(df):
    df['billed_input_tokens'] = df['input_tokens'] + df['cache_read_tokens'] + df['cache_creation_tokens']
    print(f'Total records: {len(df)}')
    print(f"Validation passed: {int(df['validation_passed'].sum())} / {len(df)}")
    print(f"Total tokens: {int(df['billed_input_tokens'].sum()):,} in / {int(df['output_tokens'].sum()):,} out")
    print(f"Mean duration: {df['duration_seconds'].mean():.2f}s")
    display(df.groupby(['dataset','target','model']).agg(
        n=('query_id','count'), passed=('validation_passed','sum'),
        mean_iter=('iterations_used','mean'),
        in_tok=('billed_input_tokens','sum'), out_tok=('output_tokens','sum')))
else:
    print('No records yet.')

Total records: 168
Validation passed: 152 / 168
Total tokens: 815,607 in / 135,310 out
Mean duration: 15.58s


n  passed  mean_iter  in_tok  out_tok
dataset target  model                                                  
ldbc    aql     claude-opus-4-8  14      14   1.000000   84841     1802
                gemma4:26b       14      14   1.071429   61532    43420
                llama3.2:latest  14      12   1.714286   96810     4716
                qwen3-coder:30b  14      14   1.000000   51364     1049
        cypher  claude-opus-4-8  14      14   1.000000   66040     1329
                gemma4:26b       14      14   1.000000   46127    23720
                llama3.2:latest  14      14   1.285714   53254     1955
                qwen3-coder:30b  14      14   1.000000   40910      833
        gremlin claude-opus-4-8  14      14   1.000000   68470     1341
                gemma4:26b       14      14   1.071429   53359    44454
                llama3.2:latest  14       3   2.571429  127989     8685
                qwen3-coder:30b  14      11   1.428571   64911     2006

Records written. Proceed to `02_behavioural_metrics.ipynb`.